In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'data').exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / 'data').exists():
    raise FileNotFoundError('Não foi possível localizar a raiz do projeto.')

for candidate in [
    PROJECT_ROOT / 'code',
    PROJECT_ROOT / 'code' / 'revenue',
    PROJECT_ROOT / 'code' / 'tmdb',
]:
    if candidate.exists() and str(candidate.resolve()) not in sys.path:
        sys.path.append(str(candidate.resolve()))

import pandas as pd

from experiment_utils import format_summary_display
from imbalance_experiment_utils import (
    ROBUST_LOSS_ARTIFACT_DIR,
    TMDB_EXTENDED_NO_TRANSFORM_ARTIFACT_DIR,
    load_results_from_artifact_dir,
    load_tmdb_extended_context,
    run_regression_model_selection,
    save_additional_table,
    save_artifact_tables,
    select_robust_model_configs_from_global_best,
    summarize_errors_by_band,
)


/home/gabriel/Faculdade/Matérias/ML/UFSJ_Aprendizado_Maquina_TP1/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# **Funções de Perda Mais Robustas na Base TMDB Estendida**

Este notebook compara versões clássicas e robustas de modelos de regressão para investigar se a assimetria de `revenue` pode ser tratada com critérios menos sensíveis a outliers, como `absolute_error`, `Huber`, `quantile` e objetivos robustos do `XGBoost`.

Os modelos mantidos aqui são escolhidos automaticamente a partir do melhor modelo global do notebook `05`. O notebook identifica a família vencedora no cenário TMDB estendido e testa apenas as variantes robustas dessa mesma família, para comparar o efeito da função de perda sem misturar arquiteturas diferentes.

In [2]:
TARGET_NAME = 'Sem transformação'
OVERWRITE_ARTIFACTS = False
SHOW_PROGRESS = True

robust_selection = select_robust_model_configs_from_global_best(
    artifact_dir=TMDB_EXTENDED_NO_TRANSFORM_ARTIFACT_DIR,
    target_name=TARGET_NAME,
)
BEST_GLOBAL_MODEL_NAME = robust_selection['best_global_model_name']
PROMISING_MODEL_NAMES = robust_selection['selected_model_names']
ROBUST_MODEL_CONFIGS_TO_RUN = robust_selection['model_configs']

ARTIFACT_DIR = ROBUST_LOSS_ARTIFACT_DIR
ERROR_ANALYSIS_DIR = ARTIFACT_DIR / 'error_analysis'
BEST_COMPARISON_PATH = ARTIFACT_DIR / 'best_robust_vs_global_best.csv'
BAND_COMPARISON_PATH = ERROR_ANALYSIS_DIR / '06_robust_losses_metricas_por_faixa.csv'


In [3]:
context = load_tmdb_extended_context()
df_movies = context['df_movies']
X = context['X']
y = context['y']
folds_df = context['folds_df']
revenue_bins = context['revenue_bins']

print(f'Base TMDB estendida: {df_movies.shape[0]} filmes | {X.shape[1]} features')
display(df_movies[['id_tmdb', 'title', 'revenue']].head())


Base TMDB estendida: 6918 filmes | 284 features


,id_tmdb,title,revenue
0,552524,Lilo & Stitch,610800000
1,950387,A Minecraft Movie,947000000
2,1257960,सिकंदर,24727058
3,574475,Final Destination Bloodlines,229314062
4,1197306,A Working Man,98652557


In [4]:
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
ERROR_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

if (
    not OVERWRITE_ARTIFACTS
    and (ARTIFACT_DIR / 'model_selection_results.csv').exists()
    and (ARTIFACT_DIR / 'model_selection_predictions.csv').exists()
    and (ARTIFACT_DIR / 'model_selection_summary.csv').exists()
):
    results_df, predictions_df, summary_df = load_results_from_artifact_dir(ARTIFACT_DIR)
else:
    results_df, predictions_df, summary_df = run_regression_model_selection(
        df_movies=df_movies,
        X=X,
        y=y,
        folds_df=folds_df,
        model_configs=ROBUST_MODEL_CONFIGS_TO_RUN,
        target_name=TARGET_NAME,
        show_progress=SHOW_PROGRESS,
    )
    save_artifact_tables(
        ARTIFACT_DIR,
        results_df=results_df,
        predictions_df=predictions_df,
        summary_df=summary_df,
    )

summary_df = summary_df.loc[summary_df['target_version'] == TARGET_NAME].copy()
global_results_df, global_predictions_df, global_summary_df = load_results_from_artifact_dir(
    TMDB_EXTENDED_NO_TRANSFORM_ARTIFACT_DIR,
)
global_summary_df = global_summary_df.loc[global_summary_df['target_version'] == TARGET_NAME].copy()

best_robust_row = summary_df.sort_values(['mean_rmse', 'mean_mae', 'model']).iloc[0]
best_global_row = global_summary_df.sort_values(['mean_rmse', 'mean_mae', 'model']).iloc[0]
best_comparison_df = pd.DataFrame([
    {
        'cenário': 'Melhor robusto',
        'modelo': best_robust_row['model'],
        'mean_rmse': best_robust_row['mean_rmse'],
        'mean_mae': best_robust_row['mean_mae'],
        'mean_r2': best_robust_row['mean_r2'],
    },
    {
        'cenário': 'Melhor global TMDB estendido',
        'modelo': best_global_row['model'],
        'mean_rmse': best_global_row['mean_rmse'],
        'mean_mae': best_global_row['mean_mae'],
        'mean_r2': best_global_row['mean_r2'],
    },
])
best_comparison_df['delta_rmse_vs_global'] = best_comparison_df['mean_rmse'] - best_global_row['mean_rmse']
best_comparison_df['delta_mae_vs_global'] = best_comparison_df['mean_mae'] - best_global_row['mean_mae']
best_comparison_df['delta_r2_vs_global'] = best_comparison_df['mean_r2'] - best_global_row['mean_r2']
save_additional_table(best_comparison_df, BEST_COMPARISON_PATH)

best_robust_predictions_df = predictions_df.loc[predictions_df['model'] == best_robust_row['model']].copy()
best_global_predictions_df = global_predictions_df.loc[
    (global_predictions_df['target_version'] == TARGET_NAME)
    & (global_predictions_df['model'] == best_global_row['model'])
].copy()

robust_band_metrics_df = summarize_errors_by_band(best_robust_predictions_df, revenue_bins)
global_band_metrics_df = summarize_errors_by_band(best_global_predictions_df, revenue_bins)
band_comparison_df = global_band_metrics_df.merge(
    robust_band_metrics_df,
    on='faixa_receita',
    suffixes=('_global', '_robusto'),
)
band_comparison_df['delta_mae_medio'] = band_comparison_df['mae_medio_robusto'] - band_comparison_df['mae_medio_global']
band_comparison_df['delta_rmse'] = band_comparison_df['rmse_robusto'] - band_comparison_df['rmse_global']
save_additional_table(band_comparison_df, BAND_COMPARISON_PATH)

print('Modelo de referência vindo do notebook 05:')
print(f'- {BEST_GLOBAL_MODEL_NAME}')
print('Variantes robustas selecionadas para esta execução:')
for model_name in PROMISING_MODEL_NAMES:
    print(f'- {model_name}')

display(format_summary_display(summary_df))
display(best_comparison_df)
display(band_comparison_df[['faixa_receita', 'mae_medio_global', 'mae_medio_robusto', 'delta_mae_medio', 'rmse_global', 'rmse_robusto', 'delta_rmse']])


Regressão robusta: 100%|██████████| 510/510 [15:20<00:00,  1.80s/ajuste, XGBoost (Pseudo-Huber) | fold 9 | melhor=-42994873195823104.0000 | refit=sim] 


Modelo de referência vindo do notebook 05:
- XGBoost Regressor
Variantes robustas selecionadas para esta execução:
- XGBoost (Squared Error)
- XGBoost (MAE)
- XGBoost (Pseudo-Huber)


MSE  \
Versão do alvo    Modelo                                           
Sem transformação XGBoost (Squared Error)  1.27e+16 (± 3.59e+15)   
                  XGBoost (MAE)            1.32e+16 (± 4.14e+15)   
                  XGBoost (Pseudo-Huber)   4.43e+16 (± 7.27e+15)   

                                                             RMSE  \
Versão do alvo    Modelo                                            
Sem transformação XGBoost (Squared Error)  111.66 mi (± 15.00 mi)   
                  XGBoost (MAE)            113.82 mi (± 18.01 mi)   
                  XGBoost (Pseudo-Huber)   209.86 mi (± 17.85 mi)   

                                                            MAE  \
Versão do alvo    Modelo                                          
Sem transformação XGBoost (Squared Error)  55.16 mi (± 2.52 mi)   
                  XGBoost (MAE)            51.58 mi (± 2.37 mi)   
                  XGBoost (Pseudo-Huber)   98.93 mi (± 2.97 mi)   

                                                         R²  
Versão do alvo    Modelo                                     
Sem transformação XGBoost (Squared Error)   0.630 (± 0.078)  
                  XGBoost (MAE)             0.620 (± 0.069)  
                  XGBoost (Pseudo-Huber)   -0.292 (± 0.051)

,cenário,modelo,mean_rmse,mean_mae,mean_r2,delta_rmse_vs_global,delta_mae_vs_global,delta_r2_vs_global
0,Melhor robusto,XGBoost (Squared Error),1.116649e+08,55157512.8,0.630175,0.0,0.0,0.0
1,Melhor global TMDB estendido,XGBoost Regressor,1.116649e+08,55157512.8,0.630175,0.0,0.0,0.0


,faixa_receita,mae_medio_global,mae_medio_robusto,delta_mae_medio,rmse_global,rmse_robusto,delta_rmse
0,Muito baixa receita,2.242120e+07,2.242120e+07,0.0,4.067001e+07,4.067001e+07,0.0
1,Baixa receita,2.853396e+07,2.853396e+07,0.0,4.425838e+07,4.425838e+07,0.0
2,Média receita,3.024786e+07,3.024786e+07,0.0,4.859167e+07,4.859167e+07,0.0
3,Alta receita,4.133001e+07,4.133001e+07,0.0,6.070357e+07,6.070357e+07,0.0
4,Muito alta receita,1.532248e+08,1.532248e+08,0.0,2.317015e+08,2.317015e+08,0.0
